In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!pip install unsloth trl datasets wandb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.2/447.2 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.7/915.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
!pip uninstall -y pyarrow datasets
!pip install pyarrow==15.0.2 datasets==2.19.1
!pip install unsloth trl wandb -q

Found existing installation: pyarrow 23.0.1
Uninstalling pyarrow-23.0.1:
  Successfully uninstalled pyarrow-23.0.1
Found existing installation: datasets 4.3.0
Uninstalling datasets-4.3.0:
  Successfully uninstalled datasets-4.3.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 78.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
ERROR: pip's dependency resolver does not currently take into

In [4]:
import logging
import transformers

# Force transformers to be quiet about deprecation warnings
transformers.utils.logging.set_verbosity_error()

# Optional: Disable the specific logger causing the crash
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)

In [5]:
%%writefile train_agent.py
import os
import torch
import re
import logging
import warnings
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import GRPOConfig, GRPOTrainer
import wandb

# 1. FIX LOG SPAM: Silence the AttentionMask warnings
logging.getLogger("transformers.modeling_attn_mask_utils").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning)

# 2. FIX DDP CRASH: Monkey-patch PyTorch's DDP wrapper to expose the config
from torch.nn.parallel import DistributedDataParallel as DDP
if not hasattr(DDP, "config"):
    DDP.config = property(lambda self: getattr(self.module, "config", None))

# 3. Load Model & LoRA
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 1024,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

# 4. Load and Format Dataset
dataset = load_dataset("openai/gsm8k", "main", split="train")

SYSTEM_PROMPT = """You are a math reasoning assistant.
Always think through problems step by step inside <think> tags.
Then give your final answer inside <answer> tags.

Example format:
<think>
Let me work through this carefully...
[your reasoning here]
</think>
<answer>42</answer>"""

def format_example(example):
    return {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": example["question"]}
        ],
        "ground_truth": example["answer"]
    }

formatted_dataset = dataset.map(format_example)
small_dataset = formatted_dataset.select(range(500))

# 5. Reward Functions
def extract_text(completion):
    if isinstance(completion, str):
        return completion
    elif isinstance(completion, list):
        return " ".join([m.get("content", "") for m in completion if isinstance(m, dict)])
    return str(completion)

def extract_answer(text):
    text = extract_text(text)
    match = re.search(r'<answer>(.*?)</answer>', text, re.DOTALL)
    if match:
        num_match = re.search(r'[\d,]+\.?\d*', match.group(1))
        if num_match:
            return num_match.group().replace(',', '')
    return None

def extract_gt_answer(gt_text):
    if "####" in str(gt_text):
        return str(gt_text).split("####")[1].strip().replace(',', '')
    return str(gt_text).strip()

def format_reward(completions, **kwargs):
    rewards = []
    for completion in completions:
        text = extract_text(completion)
        has_think = bool(re.search(r'<think>.*?</think>', text, re.DOTALL))
        has_answer = bool(re.search(r'<answer>.*?</answer>', text, re.DOTALL))
        reward = 0.5 if (has_think and has_answer) else 0.0
        rewards.append(reward)
    return rewards

def accuracy_reward(completions, **kwargs):
    ground_truths = kwargs.get("ground_truth", [])
    rewards = []
    for completion, gt in zip(completions, ground_truths):
        predicted = extract_answer(completion)
        actual = extract_gt_answer(gt)
        reward = 1.0 if (predicted and actual and predicted == actual) else 0.0
        rewards.append(reward)
    return rewards

# 6. Multi-GPU WandB Setup
if os.environ.get("LOCAL_RANK", "0") == "0":
    wandb.login(key="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") 
    wandb.init(project="my-rlvr-agent", name="day1-qwen-multigpu-fixed")
else:
    os.environ["WANDB_DISABLED"] = "true"

# 7. Training Config
training_args = GRPOConfig(
    output_dir = "/kaggle/working/grpo_gsm8k",
    num_generations = 8, 
    max_completion_length = 400, 
    learning_rate = 5e-6,
    per_device_train_batch_size = 1,
    gradient_accumulation_steps = 4,
    num_train_epochs = 1,
    fp16 = True,
    optim = "adamw_8bit",
    logging_steps = 10,
    save_steps = 100,
    report_to = "wandb" if os.environ.get("LOCAL_RANK", "0") == "0" else "none",
)

trainer = GRPOTrainer(
    model = model,
    args = training_args,
    train_dataset = small_dataset,
    reward_funcs = [accuracy_reward, format_reward],
)

if os.environ.get("LOCAL_RANK", "0") == "0":
    print("Starting Multi-GPU training! Watch WandB for reward curves.")
    
trainer.train()

Writing train_agent.py


In [6]:
!torchrun --nproc_per_node=2 train_agent.py

W0310 00:20:49.443000 102 torch/distributed/run.py:852] 
W0310 00:20:49.443000 102 torch/distributed/run.py:852] *****************************************
W0310 00:20:49.443000 102 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0310 00:20:49.443000 102 torch/distributed/run.py:852] *****************************************
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA To